# CNN+Transformer 모델 전체 비교 평가

이 노트북은 `cnn_transformer_*.pt` 계열 체크포인트를 자동으로 찾아서,
동일한 WESAD 테스트셋 기준으로 Accuracy / Classification Report / 클래스별 정확도를 비교합니다.

In [ ]:
import os
import glob
import math
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, precision_recall_fscore_support, accuracy_score
from scipy.signal import resample_poly, find_peaks, butter, filtfilt

In [ ]:
def resolve_dataset_path():
    candidates = [
        '../../../../Dataset/WESAD',
        '../../../Dataset/WESAD',
        '../../Dataset/WESAD',
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError('WESAD dataset path not found. Update candidate paths in resolve_dataset_path().')

DATASET_PATH = resolve_dataset_path()
TEST_USERS = ['S15', 'S16', 'S17']
TARGET_LABELS = {1: 0, 2: 1, 3: 2}
DOWNSAMPLE_RATE = 4
WINDOW_SIZE_SEC = 60
TEST_STRIDE_SEC = 60
LABEL_PURITY_THRESHOLD = 0.8

MODEL_DIR = '.'
MODEL_PATTERNS = ['cnn_transformer*.pt', 'CNN_Transformer*.pt']
EXTRA_MODEL_FILES = [
    '/home/binghin2/Myproject/Research/WESAD_classification/Training/Save_model/cnn_transformer_scl_scr_model.pt',
]

discovered_model_files = {f for p in MODEL_PATTERNS for f in glob.glob(os.path.join(MODEL_DIR, p))}
for extra_path in EXTRA_MODEL_FILES:
    if os.path.exists(extra_path):
        discovered_model_files.add(extra_path)
    else:
        print(f'Warning: extra model file not found -> {extra_path}')

# Remove duplicate checkpoints that point to the same real file (e.g., relative + absolute paths)
model_files = sorted({os.path.realpath(p) for p in discovered_model_files})

print(f'DATASET_PATH: {DATASET_PATH}')
print(f'Found {len(model_files)} model files')
for f in model_files:
    print(' -', os.path.basename(f))

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class CNNTransformerModel(nn.Module):
    def __init__(self, input_dim, cnn_out_channels, cnn_kernel_size, cnn_pool_size,
                 d_model, nhead, dim_feedforward, nlayers, output_dim, dropout=0.1):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, cnn_out_channels, kernel_size=cnn_kernel_size, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(cnn_pool_size)

        self.conv2 = nn.Conv1d(cnn_out_channels, cnn_out_channels, kernel_size=cnn_kernel_size, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(cnn_pool_size)

        self.cnn_to_transformer = nn.Linear(cnn_out_channels, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=nlayers)
        self.fc = nn.Linear(d_model, output_dim)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.transpose(1, 2)
        x = self.cnn_to_transformer(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x[:, -1, :]
        return self.fc(x)

class WesadDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
def _to_2d_column(signal):
    signal = np.asarray(signal)
    if signal.ndim == 1:
        return signal.reshape(-1, 1)
    return signal

def _resample_signal(signal, orig_rate, target_rate):
    signal_2d = _to_2d_column(signal)
    if orig_rate == target_rate:
        return signal_2d
    return resample_poly(signal_2d, up=target_rate, down=orig_rate, axis=0)

def decompose_eda_scl_scr(eda, fs=4, cutoff_hz=0.05, order=2):
    eda_flat = np.asarray(eda).reshape(-1).astype(float)
    nyq = 0.5 * fs
    normalized_cutoff = cutoff_hz / nyq
    b, a = butter(order, normalized_cutoff, btype='low')

    if len(eda_flat) <= (max(len(a), len(b)) * 3):
        scl = np.copy(eda_flat)
    else:
        scl = filtfilt(b, a, eda_flat)

    scr = eda_flat - scl
    return scl.reshape(-1, 1), scr.reshape(-1, 1)

def extract_bvp_features(bvp, orig_rate=64, target_rate=4):
    bvp_flat = np.asarray(bvp).flatten()
    min_distance = int(orig_rate * 0.3)
    peaks, _ = find_peaks(bvp_flat, distance=min_distance)

    times_orig = np.arange(len(bvp_flat)) / float(orig_rate)
    target_len = int(np.ceil(len(bvp_flat) * target_rate / float(orig_rate)))
    times_target = np.arange(target_len) / float(target_rate)

    if len(peaks) > 1:
        peak_times = times_orig[peaks]
        ibi = np.diff(peak_times)
        hr = 60.0 / np.clip(ibi, 1e-6, None)
        peak_times = peak_times[1:]
        hr_continuous = np.interp(times_target, peak_times, hr)
        ibi_continuous = np.interp(times_target, peak_times, ibi)
        hrv_continuous = pd.Series(ibi_continuous).rolling(window=target_rate * 10, min_periods=1).std().fillna(0.0).values
    else:
        hr_continuous = np.zeros(target_len)
        ibi_continuous = np.zeros(target_len)
        hrv_continuous = np.zeros(target_len)

    return np.concatenate([
        hr_continuous.reshape(-1, 1),
        ibi_continuous.reshape(-1, 1),
        hrv_continuous.reshape(-1, 1),
    ], axis=1)

def load_and_preprocess_data(subject_path, n_features):
    with open(subject_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    wrist_data = data['signal']['wrist']
    acc = wrist_data['ACC']
    bvp = wrist_data['BVP']
    eda = wrist_data['EDA']
    temp = wrist_data['TEMP']
    labels = data['label']

    acc_down = _resample_signal(acc, orig_rate=32, target_rate=DOWNSAMPLE_RATE)
    bvp_down = _resample_signal(bvp, orig_rate=64, target_rate=DOWNSAMPLE_RATE)
    temp_down = _resample_signal(temp, orig_rate=4, target_rate=DOWNSAMPLE_RATE)

    feature_blocks = [acc_down, bvp_down]

    if n_features == 6:
        eda_down = _resample_signal(eda, orig_rate=4, target_rate=DOWNSAMPLE_RATE)
        feature_blocks.extend([eda_down, temp_down])
    elif n_features == 9:
        eda_down = _resample_signal(eda, orig_rate=4, target_rate=DOWNSAMPLE_RATE)
        feature_blocks.extend([eda_down, temp_down])
        bvp_stats = extract_bvp_features(bvp, orig_rate=64, target_rate=DOWNSAMPLE_RATE)
        feature_blocks.append(bvp_stats)
    elif n_features == 10:
        scl_eda, scr_eda = decompose_eda_scl_scr(eda, fs=4, cutoff_hz=0.05, order=2)
        scl_down = _resample_signal(scl_eda, orig_rate=4, target_rate=DOWNSAMPLE_RATE)
        scr_down = _resample_signal(scr_eda, orig_rate=4, target_rate=DOWNSAMPLE_RATE)
        feature_blocks.extend([scl_down, scr_down, temp_down])
        bvp_stats = extract_bvp_features(bvp, orig_rate=64, target_rate=DOWNSAMPLE_RATE)
        feature_blocks.append(bvp_stats)
    else:
        raise ValueError(f'Unsupported n_features: {n_features}. Expected 6, 9, or 10.')

    label_timestamps = np.arange(len(labels)) / 700.0
    data_timestamps = np.arange(len(acc_down)) / float(DOWNSAMPLE_RATE)
    idx = np.searchsorted(label_timestamps, data_timestamps, side='left')
    idx = np.clip(idx, 0, len(labels) - 1)
    labels_down = labels[idx].astype(int)

    min_len = min(*(len(block) for block in feature_blocks), len(labels_down))
    features = np.concatenate([block[:min_len] for block in feature_blocks], axis=1)
    labels_final = labels_down[:min_len]

    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    return features_scaled, labels_final

def create_windows(features, labels, stride_sec):
    window_samples = WINDOW_SIZE_SEC * DOWNSAMPLE_RATE
    stride_samples = stride_sec * DOWNSAMPLE_RATE
    X, y = [], []

    for i in range(0, len(features) - window_samples + 1, stride_samples):
        window_features = features[i : i + window_samples]
        window_labels = labels[i : i + window_samples].astype(int)
        counts = np.bincount(window_labels)
        if counts.size == 0:
            continue

        dominant = int(counts.argmax())
        purity = counts[dominant] / float(window_samples)
        if dominant in TARGET_LABELS and purity >= LABEL_PURITY_THRESHOLD:
            X.append(window_features)
            y.append(TARGET_LABELS[dominant])

    return np.array(X), np.array(y)

test_cache = {}

def get_test_dataset(n_features):
    if n_features in test_cache:
        return test_cache[n_features]

    test_X, test_y = [], []
    for user in TEST_USERS:
        subject_path = os.path.join(DATASET_PATH, user, f'{user}.pkl')
        if not os.path.exists(subject_path):
            continue
        features, labels = load_and_preprocess_data(subject_path, n_features=n_features)
        X_user, y_user = create_windows(features, labels, stride_sec=TEST_STRIDE_SEC)
        if X_user.size > 0:
            test_X.append(X_user)
            test_y.append(y_user)

    if len(test_X) == 0:
        raise ValueError(f'No test data created for n_features={n_features}')

    X_test = np.concatenate(test_X, axis=0)
    y_test = np.concatenate(test_y, axis=0)
    test_cache[n_features] = (X_test, y_test)
    return X_test, y_test

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
class_names = ['baseline', 'stress', 'amusement']

def load_model_from_checkpoint(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)

    input_dim = int(checkpoint.get('n_features', 6))
    model = CNNTransformerModel(
        input_dim=input_dim,
        cnn_out_channels=int(checkpoint.get('cnn_out_channels', 64)),
        cnn_kernel_size=int(checkpoint.get('cnn_kernel_size', 3)),
        cnn_pool_size=int(checkpoint.get('cnn_pool_size', 2)),
        d_model=int(checkpoint.get('d_model', 64)),
        nhead=int(checkpoint.get('nhead', 4)),
        dim_feedforward=int(checkpoint.get('dim_feedforward', 256)),
        nlayers=int(checkpoint.get('nlayers', 2)),
        output_dim=int(checkpoint.get('output_dim', 3)),
        dropout=0.5,
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    return model, checkpoint, input_dim

@torch.no_grad()
def evaluate_one_model(model_path):
    model, checkpoint, n_features = load_model_from_checkpoint(model_path)
    X_test, y_test = get_test_dataset(n_features=n_features)
    test_loader = DataLoader(WesadDataset(X_test, y_test), batch_size=64, shuffle=False)

    all_preds, all_labels = [], []
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        outputs = model(sequences)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    acc = accuracy_score(all_labels, all_preds)

    precision, recall, f1, support = precision_recall_fscore_support(
        all_labels, all_preds, labels=[0, 1, 2], zero_division=0
    )
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4, zero_division=0)

    class_acc = {}
    for i, name in enumerate(class_names):
        mask = all_labels == i
        total = int(mask.sum())
        correct = int(((all_labels == i) & (all_preds == i)).sum())
        pct = (100.0 * correct / total) if total > 0 else 0.0
        class_acc[name] = {'correct': correct, 'total': total, 'pct': pct}

    print('=' * 80)
    print(f'Model: {os.path.basename(model_path)}')
    print(f'Input features: {n_features}')
    print(f'Test Accuracy: {acc * 100:.2f} %')
    print('\nClassification Report:')
    print(report)
    print('\n' + '=' * 60)
    print('각 상태별 정확도')
    print('=' * 60)
    for name in class_names:
        info = class_acc[name]
        print(f"{name.capitalize()}: {info['pct']:.2f}% ({info['correct']}/{info['total']})")
    print('=' * 60)

    return {
        'model': os.path.basename(model_path),
        'n_features': n_features,
        'accuracy_pct': acc * 100.0,
        'baseline_precision': precision[0],
        'baseline_recall': recall[0],
        'baseline_f1': f1[0],
        'stress_precision': precision[1],
        'stress_recall': recall[1],
        'stress_f1': f1[1],
        'amusement_precision': precision[2],
        'amusement_recall': recall[2],
        'amusement_f1': f1[2],
        'support_baseline': int(support[0]),
        'support_stress': int(support[1]),
        'support_amusement': int(support[2]),
    }

if len(model_files) == 0:
    raise FileNotFoundError('No cnn_transformer model files found in this directory.')

results = []
for model_path in model_files:
    try:
        results.append(evaluate_one_model(model_path))
    except Exception as exc:
        print('=' * 80)
        print(f'Model: {os.path.basename(model_path)}')
        print('평가 실패:', exc)

results_df = pd.DataFrame(results)
if len(results_df) > 0:
    results_df = results_df.sort_values(by='accuracy_pct', ascending=False).reset_index(drop=True)
    print('\n\n########## 모델 비교 요약 (Accuracy 순) ##########')
    display_cols = [
        'model', 'n_features', 'accuracy_pct',
        'baseline_f1', 'stress_f1', 'amusement_f1',
        'baseline_recall', 'stress_recall', 'amusement_recall'
    ]
    display(results_df[display_cols])
else:
    print('성공적으로 평가된 모델이 없습니다.')

In [ ]:
if 'results_df' not in globals() or len(results_df) == 0:
    raise ValueError('results_df가 비어 있습니다. 먼저 셀 6을 실행해 모델 비교를 완료하세요.')

print('\n########## 랭킹 1: 정확도(Accuracy) 우선 ##########')
rank_accuracy = results_df.sort_values(
    by=['accuracy_pct', 'amusement_recall', 'amusement_f1'],
    ascending=[False, False, False]
).reset_index(drop=True)
display(rank_accuracy[['model', 'accuracy_pct', 'amusement_recall', 'amusement_f1', 'stress_recall', 'baseline_recall']])

print('\n########## 랭킹 2: Amusement 우선 ##########')
rank_amusement = results_df.sort_values(
    by=['amusement_recall', 'amusement_f1', 'accuracy_pct'],
    ascending=[False, False, False]
).reset_index(drop=True)
display(rank_amusement[['model', 'amusement_recall', 'amusement_f1', 'accuracy_pct', 'stress_recall', 'baseline_recall']])

best_acc = rank_accuracy.iloc[0]
best_amo = rank_amusement.iloc[0]
print('\n[요약]')
print(f"정확도 최우선 추천 : {best_acc['model']} (Acc={best_acc['accuracy_pct']:.2f}%)")
print(f"Amusement 최우선 추천: {best_amo['model']} (Recall={best_amo['amusement_recall']:.4f}, F1={best_amo['amusement_f1']:.4f})")

In [ ]:
import matplotlib.pyplot as plt

if 'results_df' not in globals() or len(results_df) == 0:
    raise ValueError('results_df가 비어 있습니다. 먼저 셀 6을 실행해 모델 비교를 완료하세요.')

plot_df = results_df.copy()
plot_df['macro_f1'] = (
    plot_df['baseline_f1'] + plot_df['stress_f1'] + plot_df['amusement_f1']
) / 3.0

# Amusement 우선 점수: Recall 중심 + F1 보조
plot_df['amusement_priority_score'] = 0.7 * plot_df['amusement_recall'] + 0.3 * plot_df['amusement_f1']

rank_f1 = plot_df.sort_values('macro_f1', ascending=False).reset_index(drop=True)
rank_acc = plot_df.sort_values('accuracy_pct', ascending=False).reset_index(drop=True)
rank_amo = plot_df.sort_values('amusement_priority_score', ascending=False).reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

# 1) F1-score
axes[0].barh(rank_f1['model'], rank_f1['macro_f1'], color='#4C78A8')
axes[0].set_title('F1-score (Macro) Ranking')
axes[0].set_xlabel('Macro F1')
axes[0].invert_yaxis()

# 2) Accuracy
axes[1].barh(rank_acc['model'], rank_acc['accuracy_pct'], color='#59A14F')
axes[1].set_title('Accuracy Ranking')
axes[1].set_xlabel('Accuracy (%)')
axes[1].invert_yaxis()

# 3) Amusement 우선
axes[2].barh(rank_amo['model'], rank_amo['amusement_priority_score'], color='#E15759')
axes[2].set_title('Amusement Priority Ranking\n(0.7*Recall + 0.3*F1)')
axes[2].set_xlabel('Amusement Priority Score')
axes[2].invert_yaxis()

for ax in axes:
    ax.grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print('Top-1 by F1-score (Macro):', rank_f1.iloc[0]['model'])
print('Top-1 by Accuracy      :', rank_acc.iloc[0]['model'])
print('Top-1 by Amusement 우선:', rank_amo.iloc[0]['model'])